# Boscos aleatoris

**Optativa d'Aprenentatge automàtic · quadern de teoria 3**

Al quadern anterior vau entrenar arbres de decisió i vau veure el seu problema: un
arbre prou profund **memoritza** l'entrenament en lloc d'aprendre'l, i cau al primer
exemple que no ha vist. Avui arreglem això sense canviar d'idea, només fent-ne
**molts alhora**.

## 1. El problema que hereta

Un arbre de decisió sol té un defecte: si el deixes créixer prou, deixa de trobar
regles generals i comença a memoritzar files concretes de l'entrenament. Encerta un
100 % a classe i falla a l'examen.

La solució d'avui no és "fer un arbre més llest". És aquesta:

> Si un arbre s'equivoca d'una manera, en fem **molts** que s'equivoquin de maneres
> **diferents**, i els fem votar.

Això és un **bosc aleatori** (*random forest*): centenars d'arbres, cadascun una mica
diferent dels altres, que decideixen per majoria. La pregunta que respon aquest
quadern és **per què votar funciona** i **què cal perquè funcioni de debò**.

## 2. Per què votar funciona

Abans de tocar cap arbre, fem l'experiment amb classificadors imaginaris. Suposem
que tenim **100 classificadors**, cadascun encerta el **60 %** de les vegades, i que
**s'equivoquen de manera independent** (l'error d'un no té res a veure amb l'error
d'un altre).

Simulem-ho: generem els seus vots amb `numpy` i mirem què diu la majoria.

In [ ]:
import numpy as np

np.random.seed(42)

n_votants = 100
p_encert = 0.60
n_simulacions = 10_000

# cada fila és una simulació completa: els vots dels 100 classificadors
# True = ha encertat aquesta simulació
vots = np.random.rand(n_simulacions, n_votants) < p_encert

# la majoria encerta si més de la meitat dels vots són "True"
majoria_encerta = vots.sum(axis=1) > (n_votants / 2)
precisio_majoria = majoria_encerta.mean()

print(f"Cada classificador encerta sol: {p_encert:.0%}")
print(f"La majoria de 100 encerta:      {precisio_majoria:.1%}")

Cent votants mediocres, votant junts, es converteixen en un jutge que gairebé no
falla. Ningú ha après res de nou: només hem comptat vots.

Vegem com de ràpid puja això si anem afegint votants, d'un en un fins a 101 (sempre
un nombre senar, perquè no hi hagi empats).

In [ ]:
np.random.seed(42)

ns = list(range(1, 102, 2))  # 1, 3, 5, ..., 101
vots_base = np.random.rand(n_simulacions, max(ns)) < p_encert

precisions = []
for n in ns:
    majoria = vots_base[:, :n].sum(axis=1) > (n / 2)
    precisions.append(majoria.mean())

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(ns, precisions, marker="o", markersize=3, color="tab:blue")
plt.axhline(p_encert, color="gray", linestyle="--", label=f"un sol classificador ({p_encert:.0%})")
plt.xlabel("Nombre de votants")
plt.ylabel("Precisió de la majoria")
plt.title("Com puja la precisió a mesura que afegim votants independents")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Amb un sol votant tens un 60 % (de fet una mica menys, perquè amb un vot no hi ha
majoria a suavitzar). Amb 101 la precisió ja frega el **98 %**. Cap dels 101 sap més
que abans: només hem aprofitat que els seus errors **no coincideixen**.

**El matís honest**, i és important: tot això depèn que els errors siguin
**independents**. Si els 100 classificadors s'equivoquen exactament en els mateixos
casos —perquè en el fons són el mateix classificador repetit—, votar no arregla
res: la majoria s'equivoca igual que un de sol. El truc del bosc aleatori no és
"fer molts arbres", és **fer molts arbres que s'equivoquin de maneres diferents**.
Això és el que ve ara.

## 3. Com es fan diferents: bootstrap i atzar a les columnes

Un bosc aleatori fa diferents els seus arbres de dues maneres, totes dues basades
en l'atzar.

### 3.1 Bootstrap: cada arbre veu una mostra diferent

Cada arbre no s'entrena amb totes les files de l'entrenament, sinó amb una **mostra
del mateix mida treta a l'atzar, i amb reemplaçament** (una mateixa fila pot sortir
0, 1, 2 o més vegades). Això es diu **bootstrap**.

Com que hi ha reemplaçament, algunes files no surten mai triades. Quantes es queden
fora, de mitjana? Comprovem-ho amb una simulació abans de fer cap càlcul.

In [ ]:
np.random.seed(42)

n_files = 200
n_simulacions_boot = 2_000

fora = []
for _ in range(n_simulacions_boot):
    mostra = np.random.randint(0, n_files, size=n_files)
    files_incloses = len(set(mostra))
    fora.append(1 - files_incloses / n_files)

print(f"Mitjana de files deixades fora, per simulació: {np.mean(fora):.1%}")
print(f"Valor teòric quan n creix (1/e):                {1/np.e:.1%}")

\
Surt gairebé exactament $1/e \approx 36{,}8\%$. No és casualitat: la probabilitat
que una fila concreta **no** surti en cap dels $n$ sorteigs és

$$\left(1 - \frac{1}{n}\right)^n \xrightarrow[n\to\infty]{} \frac{1}{e} \approx 0{,}368$$

O sigui que **cada arbre veu, de mitjana, només un 63 % de les files** —repetides
algunes— i n'hi ha un 37 % que ni ha vist. Aquest 37 % és diferent per a cada arbre,
i això ja n'hi ha prou per fer-los diferents entre ells.

\
### 3.2 Atzar a les columnes: cada tall veu poques opcions

No n'hi ha prou amb variar les files. Si totes les teves columnes fossin molt bones
predint, encara que canviessis les files, **tots els arbres trobarien el mateix
primer tall** (la columna més forta guanya sempre) i tornarien a assemblar-se massa.

Per això, a cada tall, l'arbre **no pot mirar totes les columnes**: només un
subconjunt triat a l'atzar. A scikit-learn això és el paràmetre `max_features`, i
per defecte en un bosc de classificació és $\sqrt{\text{nombre de columnes}}$.

In [ ]:
n_columnes_iris = 4
print(f"Columnes disponibles a cada tall (per defecte): {int(np.sqrt(n_columnes_iris))} de {n_columnes_iris}"
      "  ->  aprox. la meitat, no totes")

Amb només dues de les quatre columnes disponibles a cada tall, un arbre que no pot
veure la columna més forta es veu obligat a fer-se fort amb una altra. Diferents
arbres acaben apostant per columnes diferents als primers nivells, i això és
exactament el que trenca la correlació entre ells: **es tornen a equivocar de
maneres diferents**, com als classificadors independents del punt 2.

## 4. Construeix un bosc a mà

Ara toca fer-ho de veritat, amb les dades d'Iris del quadern de la demostració.
Farem servir `DecisionTreeClassifier` com a peça, però **el bootstrap i la votació
els fem nosaltres**, per veure que no hi ha cap màgia amagada.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

iris = load_iris(as_frame=True)
dades = iris.frame.copy()
dades["especie"] = iris.target_names[iris.target]
dades = dades.rename(columns={
    "sepal length (cm)": "sepal_llarg",
    "sepal width (cm)": "sepal_ample",
    "petal length (cm)": "petal_llarg",
    "petal width (cm)": "petal_ample",
})

X = dades[["sepal_llarg", "sepal_ample", "petal_llarg", "petal_ample"]]
y = dades["especie"]

X_entrena, X_examen, y_entrena, y_examen = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

arbre_sol = DecisionTreeClassifier(max_depth=3, random_state=42)
arbre_sol.fit(X_entrena, y_entrena)
precisio_arbre_sol = arbre_sol.score(X_examen, y_examen)

print(f"Precisió d'un sol arbre: {precisio_arbre_sol:.1%}")

In [ ]:
from scipy import stats

n_arbres = 50
rng = np.random.RandomState(42)

X_entrena_idx = X_entrena.reset_index(drop=True)
# factoritzem les etiquetes a números perquè scipy.stats.mode treballi amb elles
y_entrena_num, especies = pd.factorize(y_entrena.reset_index(drop=True))
n_files_entrena = len(X_entrena_idx)

prediccions = np.zeros((n_arbres, len(X_examen)), dtype=int)
for i in range(n_arbres):
    # bootstrap: mostra amb reemplaçament de les files d'entrenament
    idx = rng.randint(0, n_files_entrena, size=n_files_entrena)
    Xb = X_entrena_idx.iloc[idx]
    yb = y_entrena_num[idx]

    arbre = DecisionTreeClassifier(max_depth=3, random_state=i)
    arbre.fit(Xb, yb)
    prediccions[i] = arbre.predict(X_examen)

# votació per majoria: la classe més freqüent a cada columna (cada flor)
vot_majoria_num, _ = stats.mode(prediccions, axis=0, keepdims=False)
vot_majoria = especies[vot_majoria_num]

precisio_bosc_ma = (vot_majoria == y_examen.values).mean()
print(f"Precisió del meu bosc ({n_arbres} arbres): {precisio_bosc_ma:.1%}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
rf.fit(X_entrena, y_entrena)
precisio_rf = rf.score(X_examen, y_examen)

pd.DataFrame([
    {"Model": "Un sol arbre", "Precisió": f"{precisio_arbre_sol:.1%}"},
    {"Model": f"El meu bosc ({n_arbres} arbres, fet a mà)", "Precisió": f"{precisio_bosc_ma:.1%}"},
    {"Model": "RandomForestClassifier (scikit-learn, 100 arbres)", "Precisió": f"{precisio_rf:.1%}"},
])

## 5. El cas incòmode d'Iris

Mira bé la taula anterior. **El bosc surt pitjor que l'arbre sol**, tant el que has
fet a mà com el de scikit-learn. I no és un error de codi: al quadern de la
demostració (`ML_00_demo_iris.ipynb`) ja va passar exactament el mateix, amb un
bosc de 100 arbres al 91,1 % contra el 97,8 % d'un sol arbre.

No amaguem això. **Iris és un conjunt minúscul i fàcil**: 150 flors, 4 columnes,
grups que gairebé se separen a ull. Amb un problema així no hi ha marge per
millorar, i un bosc només hi afegeix soroll: cada arbre veu una mica menys de dades
(bootstrap) i una mica menys de columnes (atzar), i amb tan poca informació de
partida, això li costa més que li ajuda.

**Els boscos guanyen quan hi ha molt per aprofitar**: moltes columnes, relacions
complicades entre elles, i soroll que confon un sol arbre. Muntem un problema així
i comparem els mateixos dos models.

In [ ]:
from sklearn.datasets import make_classification

# 20 columnes, però només 5 realment informatives; la resta són redundants o soroll
Xc, yc = make_classification(
    n_samples=2_000, n_features=20, n_informative=5, n_redundant=5,
    n_clusters_per_class=2, flip_y=0.05, random_state=42,
)
Xc_entrena, Xc_examen, yc_entrena, yc_examen = train_test_split(
    Xc, yc, test_size=0.3, random_state=42
)

arbre_dificil = DecisionTreeClassifier(random_state=42)
arbre_dificil.fit(Xc_entrena, yc_entrena)
precisio_arbre_dificil = arbre_dificil.score(Xc_examen, yc_examen)

bosc_dificil = RandomForestClassifier(n_estimators=100, random_state=42)
bosc_dificil.fit(Xc_entrena, yc_entrena)
precisio_bosc_dificil = bosc_dificil.score(Xc_examen, yc_examen)

comparativa = pd.DataFrame([
    {"Problema": "Iris (fàcil, 4 columnes)",
     "Un sol arbre": f"{precisio_arbre_sol:.1%}",
     "Bosc aleatori": f"{precisio_rf:.1%}",
     "Guanya": "l'arbre"},
    {"Problema": "make_classification (difícil, 20 columnes, soroll)",
     "Un sol arbre": f"{precisio_arbre_dificil:.1%}",
     "Bosc aleatori": f"{precisio_bosc_dificil:.1%}",
     "Guanya": "el bosc"},
])
comparativa

Amb 20 columnes, soroll i relacions que un sol tall no pot capturar, l'arbre sol
—que aquí pot créixer tan profund com vulgui— es sobreajusta i el bosc el supera
amb claredat. És la mateixa lliçó del punt 2, ara amb dades de veritat: **un model
més complicat no guanya sempre**; guanya quan el problema té prou "soroll
independent" perquè votar hi ajudi.

## 6. Quines columnes fa servir més el bosc

Un bosc aleatori et pot dir, un cop entrenat, **quines columnes ha fet servir més**
per decidir. És l'atribut `feature_importances_`.

In [ ]:
importancies = pd.Series(rf.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(7, 4))
plt.barh(importancies.index, importancies.values, color="tab:green")
plt.xlabel("Importància")
plt.title("Quines columnes fa servir més el bosc (Iris)")
plt.grid(alpha=0.3, axis="x")
plt.show()

importancies.sort_values(ascending=False)

El resultat no hauria de sorprendre't: les columnes del pètal pesen molt més que
les del sèpal, exactament les que ja vau triar a ull al quadern de la demostració.

Un avís honest: aquesta mesura té un **biaix conegut**. Tendeix a donar més
importància a columnes amb **molts valors diferents** (numèriques i contínues),
encara que no siguin les que realment expliquen millor el problema, simplement
perquè tenen més punts de tall possibles entre els quals triar. No és una mesura
perfecta, és una pista.

## 7. Pràctica

Quatre exercicis. Fes-los en ordre, que cada un fa servir el que acabes de veure.

### 7.1 Quants arbres calen?

Entrena boscos amb `n_estimators` = 1, 5, 10, 25, 50, 100, 200 sobre les dades
d'Iris (`X_entrena`, `y_entrena`, `X_examen`, `y_examen` d'aquest quadern) i dibuixa
la precisió a l'examen en funció del nombre d'arbres. A partir de quin nombre
d'arbres deixa de valer la pena afegir-ne més?

In [ ]:
# Idea:
# 1. defineix la llista de valors de n_estimators a provar
# 2. per cada valor, crea un RandomForestClassifier(n_estimators=..., random_state=42)
# 3. entrena'l amb X_entrena, y_entrena i guarda'n la precisió amb .score(X_examen, y_examen)
# 4. dibuixa una línia: eix X el nombre d'arbres, eix Y la precisió

### 7.2 Arbre contra bosc amb Wine

Carrega el conjunt `load_wine()` de `sklearn.datasets` (13 columnes de propietats
químiques de vins, 3 classes). Separa entrenament i examen, entrena un
`DecisionTreeClassifier` i un `RandomForestClassifier` amb els mateixos paràmetres
que has fet servir amb Iris, i compara'n la precisió. Qui guanya aquí, l'arbre o el
bosc? Per què creus que passa això, comparant-ho amb el punt 5?

In [ ]:
# Idea:
# 1. from sklearn.datasets import load_wine
# 2. carrega les dades i separa X (columnes) i y (etiqueta)
# 3. train_test_split igual que amb Iris
# 4. entrena un DecisionTreeClassifier i un RandomForestClassifier
# 5. compara les dues precisions i escriu la teva conclusió en un comentari

### 7.3 El preu de tants arbres

Mesura quant triga a entrenar-se un bosc de 10 arbres i un de 500, sobre les dades
que vulguis (Iris mateix serveix). Pots fer-ho amb el mòdul `time`:
`inici = time.time()` abans i `time.time() - inici` després. Amb els segons que
surtin, val la pena passar de 10 a 500 arbres per al guany de precisió que obtens?

In [ ]:
# Idea:
# import time
# per a cada nombre d'arbres (10 i 500):
#   inici = time.time()
#   entrena el RandomForestClassifier
#   segons = time.time() - inici
#   imprimeix nombre d'arbres, segons, i la precisió a l'examen

### 7.4 Trenca la independència

Torna al punt 2. Repeteix la simulació de 100 classificadors, però ara fes que
**tots copiïn el vot del primer classificador el 80 % de les vegades** (en lloc de
votar de manera totalment independent). Torna a calcular la precisió de la
majoria. Ha pujat tant com abans? Relaciona el resultat amb per què el bootstrap i
l'atzar a les columnes són necessaris.

In [ ]:
# Idea:
# 1. genera el vot del primer classificador com al punt 2 (True/False amb p=0.6)
# 2. per als altres 99: amb probabilitat 0.8, copia el vot del primer;
#    amb probabilitat 0.2, genera un vot propi i independent amb p=0.6
#    (pista: np.random.rand(...) < 0.8 et dona una màscara de "quan copiar")
# 3. calcula si la majoria encerta, per moltes simulacions
# 4. compara la precisió amb la del punt 2

## Resum

- Un arbre sol es pot sobreajustar. La solució no és fer-lo més llest: és fer-ne
  **molts i diferents, i votar**.
- Votar funciona molt bé **si els errors són independents**: 100 classificadors al
  60 % arriben al 97,2 % votant junts, i amb 101 ja freguen el 98 %.
- El bosc aleatori aconsegueix aquesta independència amb **bootstrap** (cada arbre
  veu una mostra diferent, deixant fora de mitjana un 36,8 % de les files) i amb
  **atzar a les columnes** (cada tall només veu un subconjunt).
- Un bosc no sempre guanya a un sol arbre: a Iris, petit i fàcil, **perd**. Guanya
  quan hi ha moltes columnes, relacions complicades i soroll.
- `feature_importances_` diu quines columnes ha fet servir més el bosc, amb el
  biaix que afavoreix columnes amb molts valors diferents.

### I ara què?

Fins ara, tots els models que heu vist **tallen**: decideixen amb condicionals,
sols o votant. El següent quadern trenca aquest patró: la **regressió logística**
no talla res, calcula una probabilitat amb una fórmula. És una manera de decidir
completament diferent, i és la porta d'entrada a les xarxes neuronals.